# Station Stacking v6 - KATL

Wide HRRR/GFS same-day 11am notebook for `KATL`.

This version uses source-owned v5 feature engineering, additive morning temperature trend features, and durable Optuna SQLite storage. Artifacts are written to `data/calibration/station_stacking_v6`.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

STATION_ID = "KATL"
FAST_MODE = False
OPTUNA_TRIALS = 100
STACK_OPTUNA_TRIALS = 50
OPTUNA_STARTUP_TRIALS = 30
STACK_OPTUNA_STARTUP_TRIALS = 30
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.calibration.station_stacking import (
    StationStackingConfig,
    V6_FEATURE_COLUMNS,
    missing_model_dependencies,
    run_station_year_split_experiment,
)


## V6 Feature Engineering

`feature_version="v6"` applies the source-owned v5 feature block and includes the 11am observation trend columns when present in the current-observation cache.


In [3]:
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]

V6_FEATURE_COLUMNS


['v2_recent_heat_anomaly_f',
 'v2_recent_heat_momentum_f',
 'v2_morning_warmup_to_consensus_f',
 'v2_consensus_minus_7d_actual_f',
 'v2_spread_per_warmup_f',
 'v2_humidity_warmup_interaction',
 'v3_high_so_far_above_current_f',
 'v3_remaining_warmup_from_high_so_far_f',
 'v3_high_so_far_minus_lag_1d_f',
 'v3_high_so_far_minus_7d_actual_f',
 'v3_remaining_warmup_per_spread_f',
 'v3_humidity_remaining_warmup_interaction',
 'v4_forecast_precip_total_mean_mm',
 'v4_forecast_precip_total_max_mm',
 'v4_forecast_precip_total_spread_mm',
 'v4_forecast_precip_max_1h_mean_mm',
 'v4_forecast_precip_hours_mean',
 'v4_forecast_precip_intensity_mean',
 'v4_forecast_precip_intensity_max',
 'v4_any_forecast_precip',
 'v4_all_forecast_precip',
 'v4_observed_precip_any',
 'v4_observed_precip_recent_mm_est',
 'v4_forecast_total_minus_observed_recent_mm',
 'v4_forecast_observed_precip_match',
 'v4_forecast_wet_observed_dry',
 'v4_observed_wet_forecast_dry',
 'v4_precip_humidity_interaction',
 'v4_precip_r

## Model Scores


In [4]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v6",
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v6/KATL_optuna.sqlite3')

In [5]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-10 13:37:34,916] A new study created in RDB with name: KATL_v6_base_xgboost_mae_f
[I 2026-06-10 13:37:37,748] Trial 0 finished with value: 1.783881202341321 and parameters: {'n_estimators': 799, 'learning_rate': 0.12369619597856178, 'max_depth': 6, 'min_child_weight': 2.385234757844707, 'gamma': 0.7800932022121826, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.6677615511747083}. Best is trial 0 with value: 1.783881202341321.
[I 2026-06-10 13:37:54,400] Trial 1 finished with value: 1.7699252608860097 and parameters: {'n_estimators': 1440, 'learning_rate': 0.0032515743808034223, 'max_depth': 8, 'min_child_weight': 8.23143373099555, 'gamma': 1.0616955533913808, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.2922905212920093}. Best is trial 1 with value: 1.7699252608860097.
[I 2026-06-10 13:38:01,965] Trial 2 finished with valu

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,564,1.718471,2.463242
1,validation_2024_2025,lightgbm,564,1.869635,2.670424
2,validation_2024_2025,catboost,564,1.722315,2.486888
3,validation_2024_2025,hrrr_raw,564,3.365692,4.605813
4,validation_2024_2025,gfs_raw,564,3.166768,4.362121
5,test_2026,xgboost,125,1.588664,2.125651
6,test_2026,lightgbm,125,1.575729,2.153196
7,test_2026,catboost,125,1.681534,2.304524
8,test_2026,ridge_stack,125,1.619239,2.189166
9,test_2026,hrrr_raw,125,3.563692,4.999679


## Morning Trend Coverage


In [6]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,0.0
1,observed_temp_change_last_3h_f,0.0
2,observed_morning_warmup_rate_f_per_hour,0.0
3,observed_high_so_far_change_since_9am_f,0.0


In [7]:
result.feature_columns.loc[result.feature_columns["feature"].isin(TREND_COLUMNS)]


,feature,kind


## Rounded Within 1F Accuracy


In [8]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="test_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
0,test_2026,catboost,125,78,62.400000
5,test_2026,xgboost,125,73,58.400000
4,test_2026,ridge_stack,125,72,57.600000
3,test_2026,lightgbm,125,70,56.000000
1,test_2026,gfs_raw,125,41,32.800000
2,test_2026,hrrr_raw,125,35,28.000000
6,validation_2024_2025,catboost,564,331,58.687943
10,validation_2024_2025,xgboost,564,327,57.978723
9,validation_2024_2025,lightgbm,564,312,55.319149
7,validation_2024_2025,gfs_raw,564,163,28.900709


## Version Comparison


In [9]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,lightgbm,125,1.547989,2.093915,v5
1,test_2026,ridge_stack,125,1.566708,2.118374,v5
2,test_2026,xgboost,125,1.569277,2.117086,v5
3,test_2026,lightgbm,125,1.575729,2.153196,v6
4,test_2026,xgboost,125,1.588664,2.125651,v6
...,...,...,...,...,...,...
61,validation_2024_2025,gfs_raw,541,3.557627,5.060559,v1
62,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v2
63,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v3
64,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v4


## 2026 Weather Brackets


In [10]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,125,1.588664,2.125651,42.4
1,lightgbm,125,1.575729,2.153196,42.4
2,catboost,125,1.681534,2.304524,45.6
3,ridge_stack,125,1.619239,2.189166,41.6
4,hrrr_raw,125,3.563692,4.999679,20.0
5,gfs_raw,125,2.964580,4.205912,21.6
